In [0]:
# Databricks notebook source

# ==========================================
# Monthly Finance Report
# ==========================================
# Generates analysis for the previous completed calendar month.
# Example: if run in August 2026, the report covers July 2026.

SOURCE_TABLE = "personal.finance.silver_transactions"

LAST_MONTH_TRANSACTIONS = "personal.finance.gold_last_month_transactions"
LAST_MONTH_SUMMARY = "personal.finance.gold_last_month_summary"
LAST_MONTH_KPIS = "personal.finance.gold_last_month_kpis"
LAST_MONTH_CATEGORY = "personal.finance.gold_last_month_category"
LAST_MONTH_TOP5_CATEGORY = "personal.finance.gold_last_month_top5_category"
MONTHLY_COMPARISON = "personal.finance.gold_monthly_comparison"


In [0]:
# COMMAND ----------

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

date_df = spark.sql("""
    SELECT
        CURRENT_DATE() AS run_date,
        ADD_MONTHS(TRUNC(CURRENT_DATE(), 'MONTH'), -1) AS report_start_date,
        TRUNC(CURRENT_DATE(), 'MONTH') AS report_end_date
""")

date_info = date_df.collect()[0]

run_date = date_info["run_date"]
report_start_date = date_info["report_start_date"]
report_end_date = date_info["report_end_date"]

print(f"Run date: {run_date}")
print(f"Report start date: {report_start_date}")
print(f"Report end date: {report_end_date}")


In [0]:
# COMMAND ----------

df_last_month = spark.sql(f"""
    SELECT *
    FROM {SOURCE_TABLE}
    WHERE transaction_date >= DATE('{report_start_date}')
      AND transaction_date < DATE('{report_end_date}')
""")

transaction_count = df_last_month.count()
print(f"Previous month transactions: {transaction_count}")

display(df_last_month)


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW personal.finance.gold_vw_LAST_MONTH_TRANSACTIONS AS
SELECT *
FROM {SOURCE_TABLE}
WHERE transaction_date >= DATE('{report_start_date}')
  AND transaction_date < DATE('{report_end_date}')
""")

In [0]:
# COMMAND ----------

# (
#     df_last_month.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(LAST_MONTH_TRANSACTIONS)
# )

# print(f"Created/updated: {LAST_MONTH_TRANSACTIONS}")


In [0]:
# COMMAND ----------

spark.sql(f"""
          CREATE OR REPLACE VIEW personal.finance.gold_vw_LAST_MONTH_SUMMARY AS
    SELECT
        MIN(transaction_date) AS month_start,
        MAX(transaction_date) AS month_end,
        DATE_FORMAT(MIN(transaction_date), 'MMM-yyyy') AS report_month,
        YEAR(MIN(transaction_date)) AS report_year,

        SUM(CASE WHEN transaction_type = 'INCOME' THEN amount ELSE 0 END) AS total_income,
        SUM(CASE WHEN transaction_type = 'EXPENSE' THEN amount ELSE 0 END) AS total_expense,

        SUM(CASE WHEN transaction_type = 'INCOME' THEN amount ELSE 0 END)
        -
        SUM(CASE WHEN transaction_type = 'EXPENSE' THEN amount ELSE 0 END) AS savings,
        CASE
            WHEN total_income = 0 THEN 0
            ELSE ROUND((savings / total_income) * 100, 2)
        END AS savings_rate,
        COUNT(*) AS total_transactions,
        SUM(CASE WHEN transaction_type = 'INCOME' THEN 1 ELSE 0 END) AS income_transactions,
        SUM(CASE WHEN transaction_type = 'EXPENSE' THEN 1 ELSE 0 END) AS expense_transactions
    FROM personal.finance.gold_vw_LAST_MONTH_TRANSACTIONS
""")

# display(df_summary)
# (
#     df_summary.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(LAST_MONTH_SUMMARY)
# )


In [0]:
# # COMMAND ----------

# spark.sql(f"""
#           CREATE OR REPLACE VIEW personal.finance.gold_vw_LAST_MONTH_KPI AS
#     SELECT
#         report_month,
#         report_year,
#         month_start,
#         month_end,
#         total_income,
#         total_expense,
#         savings,
#         CASE
#             WHEN total_income = 0 THEN 0
#             ELSE ROUND((savings / total_income) * 100, 2)
#         END AS savings_rate,
#         total_transactions,
#         income_transactions,
#         expense_transactions
#     FROM {LAST_MONTH_SUMMARY}
# """)

# display(df_kpis)

# (
#     df_kpis.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(LAST_MONTH_KPIS)
# )


In [0]:
# COMMAND ----------

spark.sql(f"""
          CREATE OR REPLACE VIEW personal.finance.gold_vw_LAST_MONTH_CATEGORY AS
    SELECT
        category,
        SUM(amount) AS total_expense,
        COUNT(*) AS transaction_count
    FROM personal.finance.gold_vw_LAST_MONTH_TRANSACTIONS
    WHERE transaction_type = 'EXPENSE'
    GROUP BY category
""")

# display(df_category.orderBy(col("total_expense").desc()))

# (
#     df_category.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(LAST_MONTH_CATEGORY)
# )


In [0]:
# COMMAND ----------

spark.sql(f"""
          create or replace view personal.finance.gold_vw_LAST_MONTH_TOP5_CATEGORY as
          select * from (
          select *,row_number() over (order by total_expense desc) as rank
          from personal.finance.gold_vw_LAST_MONTH_CATEGORY)
          where rank <= 5
          """)

# window_spec = Window.orderBy(col("total_expense").desc())
# df_top5 = (
#     df_category
#     .withColumn("rank", row_number().over(window_spec))
#     .filter(col("rank") <= 5)
#     .orderBy("rank")
# )

# display(df_top5)

# (
#     df_top5.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(LAST_MONTH_TOP5_CATEGORY)
# )


In [0]:
# COMMAND ----------

spark.sql(f"""
          create or replace view personal.finance.gold_vw_MONTHLY_COMPARISON as
WITH monthly AS (
    SELECT
        DATE_TRUNC('MONTH', transaction_date) AS month_date,
        SUM(CASE WHEN transaction_type = 'INCOME' THEN amount ELSE 0 END) AS income,
        SUM(CASE WHEN transaction_type = 'EXPENSE' THEN amount ELSE 0 END) AS expense
    FROM {SOURCE_TABLE}
    GROUP BY DATE_TRUNC('MONTH', transaction_date)
),
with_savings AS (
    SELECT
        month_date,
        income,
        expense,
        income - expense AS savings
    FROM monthly
),
comparison AS (
    SELECT
        MAX(CASE WHEN month_date = ADD_MONTHS(TRUNC(CURRENT_DATE(), 'MONTH'), -1) THEN income END) AS current_income,
        MAX(CASE WHEN month_date = ADD_MONTHS(TRUNC(CURRENT_DATE(), 'MONTH'), -1) THEN expense END) AS current_expense,
        MAX(CASE WHEN month_date = ADD_MONTHS(TRUNC(CURRENT_DATE(), 'MONTH'), -1) THEN savings END) AS current_savings,
        MAX(CASE WHEN month_date = ADD_MONTHS(TRUNC(CURRENT_DATE(), 'MONTH'), -2) THEN income END) AS previous_income,
        MAX(CASE WHEN month_date = ADD_MONTHS(TRUNC(CURRENT_DATE(), 'MONTH'), -2) THEN expense END) AS previous_expense,
        MAX(CASE WHEN month_date = ADD_MONTHS(TRUNC(CURRENT_DATE(), 'MONTH'), -2) THEN savings END) AS previous_savings
    FROM with_savings
)
SELECT
    current_income,
    previous_income,
    current_expense,
    previous_expense,
    current_savings,
    previous_savings,
    CASE
        WHEN previous_income = 0 OR previous_income IS NULL THEN NULL
        ELSE ROUND(((current_income - previous_income) / previous_income) * 100, 2)
    END AS income_change_pct,
    CASE
        WHEN previous_expense = 0 OR previous_expense IS NULL THEN NULL
        ELSE ROUND(((current_expense - previous_expense) / previous_expense) * 100, 2)
    END AS expense_change_pct,
    CASE
        WHEN previous_savings = 0 OR previous_savings IS NULL THEN NULL
        ELSE ROUND(((current_savings - previous_savings) / ABS(previous_savings)) * 100, 2)
    END AS savings_change_pct
FROM comparison
""")

# display(comparison_df)

# (
#     comparison_df.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(MONTHLY_COMPARISON)
# )


In [0]:
# COMMAND ----------

print("==========================================")
print("MONTHLY FINANCE REPORT COMPLETED")
print("==========================================")
print(f"Run date              : {run_date}")
print(f"Report start date     : {report_start_date}")
print(f"Report end date       : {report_end_date}")
print(f"Report month          : {report_start_date.strftime('%B %Y')}")
print(f"Transactions analyzed : {transaction_count}")
print("------------------------------------------")
print("Output tables:")
print("------------------------------------------")
# print(spark.table("personal.finance.gold_vw_last_month_transactions"))
# print(LAST_MONTH_SUMMARY)
# print(LAST_MONTH_KPIS)
# print(LAST_MONTH_CATEGORY)
# print(LAST_MONTH_TOP5_CATEGORY)
# print(MONTHLY_COMPARISON)
print("==========================================")
